In [1]:
import numpy as np
import random
import matplotlib.pyplot as plt
from typing import List, Tuple, Callable, Dict, Any

In [2]:
## Number of Constraints and parameters which is use in GA for Maximize and Minimize

class GeneticAlgorithm:
    """
    A comprehensive implementation of a genetic algorithm with various selection,
    crossover, and mutation methods.
    """
    
    def __init__(
        self,
        fitness_function: Callable,
        gene_length: int,
        population_size: int = 100,
        gene_type: str = 'binary',
        gene_bounds: Tuple[float, float] = (0, 1),
        max_generations: int = 100,
        crossover_rate: float = 0.8,
        mutation_rate: float = 0.1,
        selection_method: str = 'tournament',
        crossover_method: str = 'single_point',
        mutation_method: str = 'bit_flip',
        elitism: bool = True,
        n_elites: int = 2,
        tournament_size: int = 3,
        verbose: bool = False
    ):
        """
        Initialize the genetic algorithm.
        
        Args:
            fitness_function: Function to evaluate fitness of individuals
            gene_length: Length of the gene (number of parameters)
            population_size: Size of the population
            gene_type: Type of genes ('binary', 'real', 'integer')
            gene_bounds: Min and max values for real/integer genes
            max_generations: Maximum number of generations
            crossover_rate: Probability of crossover
            mutation_rate: Probability of mutation
            selection_method: Method for selection ('tournament', 'roulette', 'rank', 'sus')
            crossover_method: Method for crossover ('single_point', 'two_point', 'uniform')
            mutation_method: Method for mutation ('bit_flip', 'gaussian', 'random_reset', 'swap')
            elitism: Whether to use elitism (preserve best individuals)
            n_elites: Number of elite individuals to preserve
            tournament_size: Size of tournament for tournament selection
            verbose: Whether to print progress information
        """
        self.fitness_function = fitness_function
        self.gene_length = gene_length
        self.population_size = population_size
        self.gene_type = gene_type
        self.gene_bounds = gene_bounds
        self.max_generations = max_generations
        self.crossover_rate = crossover_rate
        self.mutation_rate = mutation_rate
        self.selection_method = selection_method
        self.crossover_method = crossover_method
        self.mutation_method = mutation_method
        self.elitism = elitism
        self.n_elites = n_elites
        self.tournament_size = tournament_size
        self.verbose = verbose
        
        # Initialize history
        self.best_fitness_history = []
        self.avg_fitness_history = []
        self.best_individual = None
        self.best_fitness = float('-inf')
        
        # Initialize population
        self.population = self._initialize_population()
        
    def _initialize_population(self) -> List[np.ndarray]:
        """
        Initialize population based on gene type.
        
        Returns:
            List of individuals (numpy arrays)
        """
        population = []
        
        for _ in range(self.population_size):
            if self.gene_type == 'binary':
                # Initialize with random binary values (0 or 1)
                individual = np.random.randint(0, 2, size=self.gene_length)
            elif self.gene_type == 'real':
                # Initialize with random real values within bounds
                low, high = self.gene_bounds
                individual = np.random.uniform(low, high, size=self.gene_length)
            elif self.gene_type == 'integer':
                # Initialize with random integer values within bounds
                low, high = self.gene_bounds
                individual = np.random.randint(low, high + 1, size=self.gene_length)
            else:
                raise ValueError(f"Unsupported gene type: {self.gene_type}")
            
            population.append(individual)
            
        return population
    
    def _evaluate_population(self) -> List[float]:
        """
        Evaluate fitness for all individuals in the population.
        
        Returns:
            List of fitness values
        """
        fitness_values = []
        
        for individual in self.population:
            fitness = self.fitness_function(individual)
            fitness_values.append(fitness)
            
            # Update best individual if needed
            if fitness > self.best_fitness:
                self.best_fitness = fitness
                self.best_individual = individual.copy()
                
        return fitness_values
    
   
    
    
    
    
    
    

In [3]:
## Different types of selections
def _selection(self, fitness_values: List[float]) -> List[np.ndarray]:
        """
        Select individuals for the next generation based on the chosen selection method.
        
        Args:
            fitness_values: List of fitness values for the current population
            
        Returns:
            List of selected individuals
        """
        if self.selection_method == 'tournament':
            return self._tournament_selection(fitness_values)
        elif self.selection_method == 'roulette':
            return self._roulette_wheel_selection(fitness_values)
        elif self.selection_method == 'rank':
            return self._rank_selection(fitness_values)
        elif self.selection_method == 'sus':
            return self._stochastic_universal_sampling(fitness_values)
        else:
            raise ValueError(f"Unsupported selection method: {self.selection_method}")
    
def _tournament_selection(self, fitness_values: List[float]) -> List[np.ndarray]:
        """
        Tournament selection method: randomly select tournament_size individuals 
        and pick the best one. Repeat until we have enough selected individuals.
        
        Args:
            fitness_values: List of fitness values for the current population
            
        Returns:
            List of selected individuals
        """
        selected = []
        
        # Determine number of individuals to select
        n_select = self.population_size
        
        for _ in range(n_select):
            # Randomly select tournament_size individuals
            tournament_indices = random.sample(range(self.population_size), self.tournament_size)
            
            # Find the best individual in the tournament
            best_index = tournament_indices[0]
            for idx in tournament_indices:
                if fitness_values[idx] > fitness_values[best_index]:
                    best_index = idx
            
            # Add the best individual to the selected list
            selected.append(self.population[best_index].copy())
        
        return selected
    
def _roulette_wheel_selection(self, fitness_values: List[float]) -> List[np.ndarray]:
        """
        Roulette wheel selection: select individuals with probability proportional to their fitness.
        
        Args:
            fitness_values: List of fitness values for the current population
            
        Returns:
            List of selected individuals
        """
        selected = []
        
        # Handle negative fitness values if present
        min_fitness = min(fitness_values)
        if min_fitness < 0:
            # Shift all fitness values to make them positive
            adjusted_fitness = [f - min_fitness + 1e-6 for f in fitness_values]
        else:
            adjusted_fitness = fitness_values.copy()
        
        # Calculate total fitness
        total_fitness = sum(adjusted_fitness)
        
        # Handle case where total fitness is zero
        if total_fitness == 0:
            # If all fitness values are zero, select randomly
            return random.choices(self.population, k=self.population_size)
        
        # Calculate selection probabilities
        selection_probs = [f / total_fitness for f in adjusted_fitness]
        
        # Select individuals based on their probabilities
        selected_indices = np.random.choice(
            range(self.population_size),
            size=self.population_size,
            p=selection_probs,
            replace=True
        )
        
        for idx in selected_indices:
            selected.append(self.population[idx].copy())
        
        return selected
    
def _rank_selection(self, fitness_values: List[float]) -> List[np.ndarray]:
        """
        Rank selection: assign selection probability based on the rank of individuals.
        
        Args:
            fitness_values: List of fitness values for the current population
            
        Returns:
            List of selected individuals
        """
        selected = []
        
        # Create list of (index, fitness) pairs
        indexed_fitness = list(enumerate(fitness_values))
        
        # Sort by fitness in ascending order
        indexed_fitness.sort(key=lambda x: x[1])
        
        # Assign ranks (higher rank = higher fitness)
        ranks = list(range(1, self.population_size + 1))
        
        # Calculate total rank sum
        total_rank = sum(ranks)
        
        # Calculate selection probabilities based on ranks
        selection_probs = [rank / total_rank for rank in ranks]
        
        # Convert indices back to original order
        original_indices = [idx for idx, _ in indexed_fitness]
        
        # Create mapping from original index to rank-based probability
        index_to_prob = {original_indices[i]: selection_probs[i] for i in range(self.population_size)}
        
        # Select individuals based on their rank probabilities
        selection_probs = [index_to_prob[i] for i in range(self.population_size)]
        
        selected_indices = np.random.choice(
            range(self.population_size),
            size=self.population_size,
            p=selection_probs,
            replace=True
        )
        
        for idx in selected_indices:
            selected.append(self.population[idx].copy())
        
        return selected
    
def _stochastic_universal_sampling(self, fitness_values: List[float]) -> List[np.ndarray]:
        """
        Stochastic Universal Sampling (SUS): a single random value is used to sample 
        all individuals by choosing them at evenly spaced intervals.
        
        Args:
            fitness_values: List of fitness values for the current population
            
        Returns:
            List of selected individuals
        """
        selected = []
        
        # Handle negative fitness values if present
        min_fitness = min(fitness_values)
        if min_fitness < 0:
            # Shift all fitness values to make them positive
            adjusted_fitness = [f - min_fitness + 1e-6 for f in fitness_values]
        else:
            adjusted_fitness = fitness_values.copy()
        
        # Calculate total fitness
        total_fitness = sum(adjusted_fitness)
        
        # Handle case where total fitness is zero
        if total_fitness == 0:
            # If all fitness values are zero, select randomly
            return random.choices(self.population, k=self.population_size)
        
        # Calculate step size for pointers
        step_size = total_fitness / self.population_size
        
        # Generate random starting point
        start = random.uniform(0, step_size)
        
        # Generate pointers
        pointers = [start + i * step_size for i in range(self.population_size)]
        
        # Select individuals
        i = 0
        cumulative_fitness = adjusted_fitness[0]
        
        for pointer in pointers:
            # Find individual whose cumulative fitness is greater than the pointer
            while cumulative_fitness < pointer:
                i = (i + 1) % self.population_size
                cumulative_fitness += adjusted_fitness[i]
            
            selected.append(self.population[i].copy())
        
        return selected

In [4]:
## Crossovers
def _crossover(self, parents: List[np.ndarray]) -> List[np.ndarray]:
        """
        Perform crossover on pairs of parents to create offspring.
        
        Args:
            parents: List of parent individuals
            
        Returns:
            List of offspring individuals
        """
        offspring = []
        
        # Shuffle parents to create random pairs
        random.shuffle(parents)
        
        # Perform crossover on pairs of parents
        for i in range(0, len(parents), 2):
            # If we have an odd number of parents, add the last one without crossover
            if i + 1 >= len(parents):
                offspring.append(parents[i].copy())
                continue
            
            parent1 = parents[i]
            parent2 = parents[i + 1]
            
            # Decide whether to perform crossover based on crossover rate
            if random.random() < self.crossover_rate:
                child1, child2 = self._perform_crossover(parent1, parent2)
                offspring.append(child1)
                offspring.append(child2)
            else:
                # No crossover, just copy parents
                offspring.append(parent1.copy())
                offspring.append(parent2.copy())
        
        return offspring
    
def _perform_crossover(self, parent1: np.ndarray, parent2: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """
        Perform the actual crossover operation based on the chosen method.
        
        Args:
            parent1: First parent individual
            parent2: Second parent individual
            
        Returns:
            Tuple of two offspring individuals
        """
        if self.crossover_method == 'single_point':
            return self._single_point_crossover(parent1, parent2)
        elif self.crossover_method == 'two_point':
            return self._two_point_crossover(parent1, parent2)
        elif self.crossover_method == 'uniform':
            return self._uniform_crossover(parent1, parent2)
        else:
            raise ValueError(f"Unsupported crossover method: {self.crossover_method}")
    
def _single_point_crossover(self, parent1: np.ndarray, parent2: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """
        Single-point crossover: choose a random point and swap segments of parents.
        
        Args:
            parent1: First parent individual
            parent2: Second parent individual
            
        Returns:
            Tuple of two offspring individuals
        """
        # Create copies of parents
        child1 = parent1.copy()
        child2 = parent2.copy()
        
        # Choose a random crossover point
        point = random.randint(1, self.gene_length - 1)
        
        # Swap segments
        child1[point:] = parent2[point:]
        child2[point:] = parent1[point:]
        
        return child1, child2
    
def _two_point_crossover(self, parent1: np.ndarray, parent2: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """
        Two-point crossover: choose two random points and swap the middle segment.
        
        Args:
            parent1: First parent individual
            parent2: Second parent individual
            
        Returns:
            Tuple of two offspring individuals
        """
        # Create copies of parents
        child1 = parent1.copy()
        child2 = parent2.copy()
        
        # Choose two random points
        point1 = random.randint(1, self.gene_length - 2)
        point2 = random.randint(point1 + 1, self.gene_length - 1)
        
        # Swap middle segments
        child1[point1:point2] = parent2[point1:point2]
        child2[point1:point2] = parent1[point1:point2]
        
        return child1, child2
    
def _uniform_crossover(self, parent1: np.ndarray, parent2: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """
        Uniform crossover: randomly select genes from either parent for each position.
        
        Args:
            parent1: First parent individual
            parent2: Second parent individual
            
        Returns:
            Tuple of two offspring individuals
        """
        # Create copies of parents
        child1 = parent1.copy()
        child2 = parent2.copy()
        
        # Generate random mask to determine which genes to swap
        mask = np.random.randint(0, 2, size=self.gene_length).astype(bool)
        
        # Apply mask to swap genes
        temp = child1[mask]
        child1[mask] = child2[mask]
        child2[mask] = temp
        
        return child1, child2

In [5]:
## Mutations 
def _mutation(self, offspring: List[np.ndarray]) -> List[np.ndarray]:
        """
        Apply mutation to offspring individuals.
        
        Args:
            offspring: List of offspring individuals
            
        Returns:
            List of mutated offspring individuals
        """
        mutated = []
        
        for individual in offspring:
            # Create a copy of the individual
            mutated_individual = individual.copy()
            
            # Apply mutation based on mutation rate and method
            if self.mutation_method == 'bit_flip':
                self._bit_flip_mutation(mutated_individual)
            elif self.mutation_method == 'gaussian':
                self._gaussian_mutation(mutated_individual)
            elif self.mutation_method == 'random_reset':
                self._random_reset_mutation(mutated_individual)
            elif self.mutation_method == 'swap':
                self._swap_mutation(mutated_individual)
            else:
                raise ValueError(f"Unsupported mutation method: {self.mutation_method}")
            
            mutated.append(mutated_individual)
        
        return mutated
    
def _bit_flip_mutation(self, individual: np.ndarray) -> None:
        """
        Bit flip mutation: flip bits with probability equal to mutation rate.
        Best used with binary genes.
        
        Args:
            individual: Individual to mutate (modified in-place)
        """
        for i in range(self.gene_length):
            if random.random() < self.mutation_rate:
                if self.gene_type == 'binary':
                    # Flip bit (0 to 1 or 1 to 0)
                    individual[i] = 1 - individual[i]
                else:
                    # For non-binary genes, use random reset instead
                    self._reset_gene(individual, i)
    
def _gaussian_mutation(self, individual: np.ndarray) -> None:
        """
        Gaussian mutation: add random value from normal distribution.
        Best used with real-valued genes.
        
        Args:
            individual: Individual to mutate (modified in-place)
        """
        for i in range(self.gene_length):
            if random.random() < self.mutation_rate:
                if self.gene_type == 'real':
                    # Add Gaussian noise
                    std_dev = (self.gene_bounds[1] - self.gene_bounds[0]) * 0.1
                    individual[i] += np.random.normal(0, std_dev)
                    
                    # Clip to bounds
                    individual[i] = max(min(individual[i], self.gene_bounds[1]), self.gene_bounds[0])
                elif self.gene_type == 'integer':
                    # Add Gaussian noise and round to nearest integer
                    std_dev = (self.gene_bounds[1] - self.gene_bounds[0]) * 0.1
                    individual[i] += np.random.normal(0, std_dev)
                    individual[i] = int(round(individual[i]))
                    
                    # Clip to bounds
                    individual[i] = max(min(individual[i], self.gene_bounds[1]), self.gene_bounds[0])
                else:
                    # For binary genes, use bit flip instead
                    individual[i] = 1 - individual[i]
    
def _random_reset_mutation(self, individual: np.ndarray) -> None:
        """
        Random reset mutation: replace gene with a random value within bounds.
        
        Args:
            individual: Individual to mutate (modified in-place)
        """
        for i in range(self.gene_length):
            if random.random() < self.mutation_rate:
                self._reset_gene(individual, i)
    
def _reset_gene(self, individual: np.ndarray, index: int) -> None:
        """
        Reset a single gene to a random value based on gene type.
        
        Args:
            individual: Individual containing the gene
            index: Index of the gene to reset
        """
        if self.gene_type == 'binary':
            individual[index] = random.randint(0, 1)
        elif self.gene_type == 'real':
            individual[index] = random.uniform(self.gene_bounds[0], self.gene_bounds[1])
        elif self.gene_type == 'integer':
            individual[index] = random.randint(self.gene_bounds[0], self.gene_bounds[1])
    
def _swap_mutation(self, individual: np.ndarray) -> None:
        """
        Swap mutation: swap two random genes.
        
        Args:
            individual: Individual to mutate (modified in-place)
        """
        if random.random() < self.mutation_rate and self.gene_length > 1:
            # Choose two distinct positions
            i, j = random.sample(range(self.gene_length), 2)
            
            # Swap genes
            individual[i], individual[j] = individual[j], individual[i]